# Monday, November 9th, 2026

Last week, we discussed using the mean and/or median filters to remove salt and pepper noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

We worked with some sample images from the course webpage, and converted them to grayscale.

In [ ]:
circuit = plt.imread('circuit.png')
gray_circuit = np.mean(circuit, axis=2) # Convert the RGB circuit array to 2D grayscale

In [ ]:
padded_circuit = get_padded_img(gray_circuit, 3)

We then added salt and pepper noise to these images.

In [ ]:
noisy_circuit = gray_circuit.copy()   # Initialize noisy_img as a 2D grayscale array
nrows, ncols = noisy_circuit.shape

r = np.random.random((nrows, ncols))  # Array of the same shape as noisy_img full of random floats
                                      # between 0 and 1.

salt_mask =   r > .9 # Identify 10% of entries that will be hit with salt noise
pepper_mask = r < .1 # Identify 10% of entries that will be hit with pepper noise

noisy_circuit[salt_mask]   = 1 # Add salt noise
noisy_circuit[pepper_mask] = 0 # Add pepper noise

We then applied the mean/median filter using `3` by `3` grids and ignoring any edge pixels.

In [ ]:
median_filtered_circuit = noisy_circuit.copy()
mean_filtered_circuit = noisy_circuit.copy()
nrows, ncols = noisy_circuit.shape

for i in range(1,nrows-1):     # For every row other than the first and last
    for j in range(1,ncols-1): # For every column other than the first and last
        grid = noisy_circuit[i-1:i+2, j-1:j+2]     # Build a 3x3 grid centered at pixel i,j
        grid_median = np.median(grid)
        grid_mean = np.mean(grid)
        mean_filtered_circuit[i,j] = grid_mean     # Set i,j-th filtered pixel to mean of the 3x3 grid
        median_filtered_circuit[i,j] = grid_median # Set i,j-th filtered pixel to median of the 3x3 grid

In [ ]:
plt.figure(figsize=(10,8))

plt.subplot(2,2,1)
plt.imshow(gray_circuit, cmap='gray', vmin=0, vmax=1)
plt.title('Original image')

plt.subplot(2,2,2)
plt.imshow(noisy_circuit, cmap='gray', vmin=0, vmax=1)
plt.title('Noisy image')

plt.subplot(2,2,3)
plt.imshow(mean_filtered_circuit, cmap='gray', vmin=0, vmax=1)
plt.title('Mean (3x3) filtered image')

plt.subplot(2,2,4)
plt.imshow(median_filtered_circuit, cmap='gray', vmin=0, vmax=1)
plt.title('Median (3x3) filtered image')

plt.tight_layout()

We then discussed how to modify this approach to deal with filtering pixels along the edge of the image. We accomplished this by placing the original array into a larger array (e.g. with padding).

### Handling different grid sizes

So far, we've been applying the mean/median filters using specifically a `3` by `3` grid centered on each pixel. What if we wan to use grids of different sizes, say `s=3`, `s=5`, `s=7`,...?

Let's think through how we would change our code to treat `5` by `5` grids. For now, let's again ignore the filtering of edge pixels.

What about `7` by `7` grids?

*Note*: The grid size `s` and the pad size `pad` are related by `pad = (s-1)//2` or `s = 2*pad + 1`.

**Exercise:** Modify the examples above to take in a parameter `s`, which is an odd integer, and apply the mean/median filter using an `s` by `s` grid. You can skip filtering any rows/columns along the edge as necessary.

### Putting things together

We've discussed how to deal with edge pixels (for the special case of `3` by `3` grids) and how to deal with varying grid sizes. We want to be able to handle both at the same time.

**Exercise:** Combine the two discussions (dealing with different grid sizes and dealing with edge pixels) to write functions `mean_filter(img, s)` and `median_filter(img,s)` that apply the mean/median filters to an input 2D array `img` using `s` by `s` grids, including all edge pixels.

## Remaining tasks to be addressed for Project 4

The exercises over the last week have asked you to create functions that accomplish the tasks that we've worked on in class. Namely, we want to have the following functions available:
 - The `sp.noise` function should take in a grayscale image array and a `noise` parameter and create a copy of the image array with salt and pepper noise added (with approximately $100\cdot$`noise`\% of the pixels being hit with salt/pepper noise).
 - The `get_padded_img` function should take in a grayscale array and a desired `pad` size  and create a padded array with `pad` additional rows at the top/bottom and `pad` additional columns at the left/right of the image array.
 - The `mean_filter` function should take in a grayscale image array and apply the mean filter, accounting for edge pixels (using the `get_padded_img` function).
 - The `median_filter` function should take in a grayscale image array and apply the median filter, accounting for edge pixels (using the `get_padded_img` function).
 - You may choose to combine the `mean_filter` and `median_filter` functions into a single function, which takes in an additional argument `filter_func` that controls the type of filter (e.g. `filter_func = np.mean` or `filter_func = np.median`).


Once you have working mean/median filters that deal with edge pixels using any odd-sized grid, you can explore how well they work on images with varying levels of noise. Does the noise level affect which filter works better? What about the grid size? Does the "best" grid size change when the amount of noise changes?

Additionally, try to think of other ways the images could be filtered. You can start from the mean/median filters and think of ways to modify or improve them, or come up with something new altogether.

## Speed - "NumPy-ifying" the process

For large images, the median filter code above is somewhat slow. It would be nice if we could use NumPy to avoid the nested `for` loop iteration through every row and column.

For reference, let's time how long it takes to apply the median filter to an image. I will use the sample code at the top of the notebook, but you should use your `median_filter` function instead.

*Idea*: Let's construct nine versions of our array. In particular, we want the following variations of the `noisy_img` array:
1. The `noisy_img` array,
2. The `noisy_img` array shifted down by one pixel,
3. The `noisy_img` array shifted up by one pixel,
4. The `noisy_img` array shifted left by one pixel,
5. The `noisy_img` array shifted right by one pixel,
6. The `noisy_img` array shifted up by one pixel and left by one pixel,
7. The `noisy_img` array shifted up by one pixel and right by one pixel,
8. The `noisy_img` array shifted down by one pixel and left by one pixel, and
9. The `noisy_img` array shifted down by one pixel and right by one pixel.

Then if we collect the `[i,j]`th entry of each of these shifted versions, we have all the values that makeup our `3` by `3` grid to compute a median.

How can we construct each of these shifted versions? We will again want to use a padded version of our array to make room for each of these shifts. We can then just take various slices of the `padded_img` array to construct these shifted versions.

Now that we have all of these shifted versions, we need some way to compute medians element-wise through these nine shifts. Let's create a new $4$-dimesional array, `shifts`, with the following structure:
 - Axis `0` corresponds to the pixel row in the original image.
 - Axis `1` corresponds to the pixel column in the original image.
 - Axis `2` corresponds to the row within the `3` by `3` grid of neighboring pixels in the original image.
 - Axis `3` corresponds to the column within the `3` by `3` grid of neighboring pixels in the original image.

In other words, `shifts` will have shape `(nrows, ncols, 3, 3)` and `shifts[i,j,m,n]` will store the pixel in the `[m,n]` position of `grid`, where `grid` is the `3` by `3` grid of pixels centered on pixel `[i,j]` of the original image.

With this new structure, we now want to compute the mean or median of each `3` by `3` grid of neighboring pixels, separately for each pixel in the original image. That is, we want to compute `np.mean(shifts[i,j])` separately for each pixel in position `[i,j]` of the original image.

Recall: The `np.mean` and `np.median` functions take in an optional argument `axis` that specify through which axes the mean/median should be computed. In our case, we want to compute the mean/median through axes `2` and `3` (i.e. the two axes that correspond to the row/column position in each `3` by `3` grid of neighboring pixels).

Let's time it to see how this compares.

We see that this is *way* faster than our previous approach!

*Note*: In this implementation, we manually wrote out each of the nine shifted arrays that were used. Can we do this in a more algorithmic way? That is, can we have Python automatically generate these shifts? One idea is to again use a pair of nested `for` loops, but this time we iterate through lists of horizontal and vertical shifts (instead of rows and columns).

*Note*: This implementation is hard-coded to use `3` by `3` grid sizes. Can we modify it to work with any odd-sized grid?

# Working with text files in Python

We've seen several functions that can be used for processing text (`str`) data. For example, things like the `.split`, `.replace`, and `.join` methods can be used to manipulate strings. Today, we will look more into working with text data in Python. 

[Project Gutenberg](https://www.gutenberg.org/) is a repository containing over 75,000 free eBooks that are in the public domain. The code cell below will download the text from *Frankenstein; Or, The Modern Prometheus* by Mary Shelley and save it as a text file `frankenstein.txt`.

*Note:* Feel free to browse through the Project Gutenberg library and select any other eBook of your choice. Just make sure to download a plain-text version of the eBook (not PDF, EPUB, HTML, or any other formats).

In [ ]:
import requests

data = requests.get('https://www.gutenberg.org/cache/epub/42324/pg42324.txt')
with open('frankenstein.txt','wb') as f:
    f.write(data.content)

To get started, we will need to be able to load the downloaded text file into Python. The `open` function can be used in Python to open a file for reading or writing. For example, `open('frankenstein.txt', 'r')` will open the `frankenstein.txt` file for reading (signified by the argument `'r'`). Once a file is opened, we can use the `.read()` method to read the contents of the file into a string.

This particular text file uses UTF-8 encoding, which is not the default encoding that the `open` function expects for text data. We can include an optional argument `encoding='utf-8-sig'` when opening the file to account for this non-default encoding.

In [ ]:
f = open('frankenstein.txt', 'r', encoding='utf-8-sig') # Opens the file for reading
text = f.read()                                         # Reads the contents into a string

print(text[2000:3000])

When opening a file in Python, the file remains open to Python until it is closed using the `.close` method. If we forget to close the file, it is possible for undesirable things to happen (for example, the file could become corrupted). 

In [ ]:
f.close()                                            # Closes the file

To ensure that we do not forget to close the file, we can use the `with` construct to have Python automatically close the file when we are done reading from it. To use this `with` construct, we assign a temporary variable name to an opened file. We then carry out any desired operations on this temporary variable in an indented block. Once the code exits the indented block, the file is closed.

The code below demonstrates how this works.

In [ ]:
with open('frankenstein.txt', 'r', encoding='utf-8-sig') as f:  # Opens the file for reading
    text = f.read()                                            # Reads the content into a string

# The file `f` is now closed, since we have exited the `with` block.

print(text[2000:3000])

## Analyzing text data

Now that we have some text to work with, what questions can we ask/answer? Suppose that we want to explore the frequency of the words that appear in *Frankenstein*.

First, it will be helpful to obtain a list of the words that appear in Frankenstein. As a starting point, we can use the `.split` method to separate the `text` string into a list of substrings separated by any amount of white space. This will (generally) give us the list of words in `text`.

**Exercise:** Construct a dictionary `word_count_dict` whose keys are words and whose values count how many times that word appears in *Frankenstein*.

To construct this dictionary:
 - Look through every word in *Frankenstein*.
 - If the word is not in our dictionary, add it to the dictionary with a value of `1`.
   - We can test whether a dictionary `word_count_dict` contains a key `word` using the Boolean expression `word in word_count_dict`. This will be `True` if `word` is a defined key in `word_count_dict` and `False` otherwise.
 - If the word is in the dictionary, increment the value associated to that word by `1`.

How many times does the word `'monster'` appear in the text?

We've generated a dictionary mapping words to their counts, but which word has the highest count? We can try using the `max` function to find the most frequently appearing word.

What is happening here? Is the symbol above truly the most frequently appearing word in Frankeinstein?

By default, taking the maximum of a dictionary returns the maximum of the keys of that dictionary. In this case, the keys are strings, so it returns the last string in lexicographic (i.e. alphabetic) order. This is illustrated in the simple example below.

In [ ]:
'Hello' < 'goodbye'

In [ ]:
my_dict = {'a': 5,
           'b': 3,
           'c': 1}

In [ ]:
max(my_dict)

In order to find the most frequently appearing word in `word_count_dict`, we'd rather take the maximum of the values (that is, of the word counts). We can use `word_count_dict.values()` to get a "list" of the values, and then take the maximum using the `max` function.

What we'd really like to do is to find the "maximum" key, where we measure the "size" of a key by its associated value.

## Finding maximums/minimums with a custom key

When using `max` to find the maximum value of an iterable, we can optionally include an argument `key` (a function) that tells the `max` function how to items are to be compared. This `key` function must be able to take in any of the items in the iterable, and return some type of data that can be sorted (like integers, floats, strings, etc.).

For example, suppose we have a list of strings and we want to find the string that is the longest. We could use `key=len` inside the `max` function to find the longest string.

We can also supply a custom key when using the `min` function to find minimums.

Note: If there are multiple items that are maximums or minimums, Python will return the first of these items that it encountered in the list. In the example above, both `'One'` and `'Two'` have the minimum length of `3`. Python returns `'One'` since it appears before `'Two'` in the list.

Let's define our own function that can be used to compare the strings in `my_list`.

Returning to the *Frankenstein* text, we want to find the "maximum" word, where the size of each word is given by the number of times that word appears in the text. 

**Exercise:** Write a function `word_count_key` that takes in a string `word` and returns the number of times that `word` appears in *Frankenstein*. You should use the `word_count_dict` that was defined earlier.

**Exercise:** Use the `word_count_key` function to find the most frequently appearing word in *Frankenstein*, along with the number of times that the word appears.

## Sorting

The `sorted` function can take in an iterable structure (e.g. list, dictionary, etc.) and return a sorted list of those items.

Just like the `max` and `min` functions, we can supply a `key` input argument to change the way that the items are sorted.

By default, the `sorted` function will sort in ascending order (i.e. from smallest to largest). We can switch to descending order using the optional argument `reverse=True`.

Note: When sorting a dictionary, the `sorted` function will return a list of sorted keys (and will drop the dictionary structure and associated values).

**Exercise:** Use the `sorted` function along with the `word_count_key` function to sort the keys of `word_count_dict` from most frequently appearing to least frequently appearing. Then print out the 10 most frequently appearing words along with their corresponding word counts.

Suppose we want to know where a word lies in the `sorted_words` list, i.e. what its word count ranking is? The `.index` method to find where an item appears in a list.

For example, `sorted_words.index('monster')` will return the index `i` such that `sorted_words[i] = 'monster'`.